# 04 - Install NBIS and verify the matching pipeline

CPU only. Run cells in order on Colab after notebook 03. This is an integration/smoke test, NOT a baseline accuracy estimate. No thresholds are chosen, no model is trained and no validation/external-test images are used.

Tools: official NIST NBIS 5.0.0, MINDTCT minutiae extraction and BOZORTH3 matching. Source: https://nigos.nist.gov/nist/nbis/nbis_v5_0_0.zip . NIST documentation: https://www.nist.gov/services-resources/software/nist-biometric-image-software-nbis .

We build locally on the runtime, retain logs/source archive on Drive, and record source/binary hashes. The build restores executable permissions lost during ZIP extraction and uses GCC -fcommon for this legacy C code. No upstream source algorithms are modified.

Three images from training FVC2000 DB1 (500 dpi) are used: two distinct impressions of one finger and one impression of another finger. Conversion is lossless grayscale PNG, no enhancement or resizing. Default MINDTCT/BOZORTH3 coordinates are used together (no -m1, no -b). A self-match checks plumbing; genuine/impostor smoke scores do not establish recognition accuracy.

Outputs: MyDrive/FingerprintResearch/baselines/nbis_smoke_v1/<UTC run ID>. Minutiae are biometric data; do not share generated reports/templates without appropriate permission. Local files disappear when the runtime is deleted.

In [ ]:
import hashlib
import json
import platform
import re
import shutil
import subprocess
import zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from google.colab import drive

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
LOCAL_ROOT = Path('/content/fingerprint_research')
DATA_ROOT = LOCAL_ROOT / 'data'
PROTOCOL_DIR = PROJECT_ROOT / 'protocols' / 'fvc_split_v1'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
REPORT_DIR = PROJECT_ROOT / 'baselines' / 'nbis_smoke_v1' / RUN_ID
WORK_DIR = LOCAL_ROOT / 'nbis_smoke' / RUN_ID
for directory in (REPORT_DIR, WORK_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

protocol_path = PROTOCOL_DIR / 'protocol.json'
protocol = json.loads(protocol_path.read_text(encoding='utf-8'))
if protocol.get('version') != 'fvc_split_v1':
    raise RuntimeError('Unexpected split protocol version.')
for name, expected in protocol['file_sha256'].items():
    if file_digest(PROTOCOL_DIR / name) != expected:
        raise RuntimeError(f'Frozen protocol file changed: {name}')
locked = pd.read_csv(PROTOCOL_DIR / 'manifest_splits.csv', keep_default_na=False)
print('Verified frozen protocol:', PROTOCOL_DIR)
print('Reports for this run:', REPORT_DIR)

In [ ]:
NBIS_URL = 'https://nigos.nist.gov/nist/nbis/nbis_v5_0_0.zip'
NBIS_SHA256 = '0adf8ab0f6b0e4208de50ca00ba21d3d77112ecd66288757ddfed21f6bee92c3'
NBIS_BYTES = 52595795
ARCHIVE_DIR = PROJECT_ROOT / 'archives'
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
NBIS_ARCHIVE = ARCHIVE_DIR / 'nbis_v5_0_0.zip'
missing_tools = [name for name in ('curl', 'gcc', 'make', 'sh') if shutil.which(name) is None]
if missing_tools:
    raise RuntimeError(f'Missing build tools: {missing_tools}. Install required tools before retrying.')
if not NBIS_ARCHIVE.exists():
    partial = LOCAL_ROOT / 'nbis_v5_0_0.zip.part'
    if not partial.exists() or partial.stat().st_size != NBIS_BYTES:
        subprocess.run([
            'curl', '--fail', '--location', '--continue-at', '-',
            '--connect-timeout', '30', '--max-time', '600',
            '--output', str(partial), NBIS_URL,
        ], check=True)
    if partial.stat().st_size != NBIS_BYTES or file_digest(partial) != NBIS_SHA256:
        raise RuntimeError('NBIS download integrity failure. Partial file retained; do not build it.')
    pending = NBIS_ARCHIVE.with_suffix('.zip.part')
    shutil.copyfile(partial, pending)
    if file_digest(pending) != NBIS_SHA256:
        raise RuntimeError('NBIS Drive copy checksum mismatch.')
    pending.replace(NBIS_ARCHIVE)
    partial.unlink()
if NBIS_ARCHIVE.stat().st_size != NBIS_BYTES or file_digest(NBIS_ARCHIVE) != NBIS_SHA256:
    raise RuntimeError('Cached NBIS archive differs from the verified release.')
print('Official NBIS source verified:', NBIS_SHA256)

In [ ]:
BUILD_ROOT = LOCAL_ROOT / 'tools' / 'nbis_5_0_0'
SOURCE_ROOT = BUILD_ROOT / 'source'
PREFIX = BUILD_ROOT / 'install'
SOURCE = SOURCE_ROOT / 'Rel_5.0.0'
MINDTCT = PREFIX / 'bin' / 'mindtct'
BOZORTH3 = PREFIX / 'bin' / 'bozorth3'
STAMP = BUILD_ROOT / 'verified_build.json'
build_commands = [
    ['sh', 'setup.sh', str(PREFIX), '--without-X11', '--without-OPENJP2', '--64'],
    ['make', 'config'],
    ['make', 'it', 'CC=gcc -fcommon'],
    ['make', 'install', 'LIBNBIS=no', 'CC=gcc -fcommon'],
]
build_signature = {'source_sha256': NBIS_SHA256, 'commands': build_commands}
if STAMP.exists():
    build_record = json.loads(STAMP.read_text(encoding='utf-8'))
    if build_record['signature'] != build_signature:
        raise RuntimeError('Cached NBIS build configuration differs; do not reuse it.')
    for binary in (MINDTCT, BOZORTH3):
        if not binary.is_file() or file_digest(binary) != build_record['binary_sha256'][binary.name]:
            raise RuntimeError(f'Cached NBIS binary changed or is missing: {binary}')
    print('Reusing verified local NBIS build.')
else:
    SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
    PREFIX.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(NBIS_ARCHIVE) as archive:
        for member in archive.infolist():
            path = PurePosixPath(member.filename)
            if path.is_absolute() or '..' in path.parts or '\\' in member.filename:
                raise RuntimeError(f'Unsafe source archive path: {member.filename}')
        archive.extractall(SOURCE_ROOT)
    for path in SOURCE.rglob('*'):
        if path.is_file() and (path.name == 'configure' or path.suffix == '.sh'):
            path.chmod(path.stat().st_mode | 0o111)
    build_log = WORK_DIR / 'build.log'
    with build_log.open('w', encoding='utf-8') as output:
        for command in build_commands:
            print('Building:', ' '.join(command), flush=True)
            result = subprocess.run(command, cwd=SOURCE, stdout=output, stderr=subprocess.STDOUT)
            output.flush()
            if result.returncode:
                shutil.copyfile(build_log, REPORT_DIR / 'build.log')
                print('\n'.join(build_log.read_text(errors='replace').splitlines()[-60:]))
                raise RuntimeError(f'NBIS build failed with code {result.returncode}. See saved build.log.')
    shutil.copyfile(build_log, REPORT_DIR / 'build.log')
    build_record = {
        'signature': build_signature,
        'binary_sha256': {binary.name: file_digest(binary) for binary in (MINDTCT, BOZORTH3)},
        'compiler': subprocess.run(['gcc', '--version'], check=True, capture_output=True, text=True).stdout,
        'platform': platform.platform(),
    }
    STAMP.write_text(json.dumps(build_record, indent=2), encoding='utf-8')
versions = {}
for binary in (MINDTCT, BOZORTH3):
    result = subprocess.run([str(binary), '-version'], check=True, capture_output=True, text=True)
    versions[binary.name] = result.stdout + result.stderr
    print(binary.name, versions[binary.name].strip())
(REPORT_DIR / 'build_provenance.json').write_text(
    json.dumps({**build_record, 'versions': versions}, indent=2), encoding='utf-8')

In [ ]:
candidates = locked[
    locked['split'].eq('train') & locked['dataset'].eq('FVC2000') & locked['db'].eq('DB1')
].sort_values(['identity_key', 'impression_id'])
ids = sorted(candidates['identity_key'].unique())
if len(ids) < 2:
    raise RuntimeError('Need two training fingers in FVC2000 DB1 for smoke test.')
first = candidates[candidates['identity_key'].eq(ids[0])].drop_duplicates('sha256').head(2)
other = candidates[candidates['identity_key'].eq(ids[1])].head(1)
samples = pd.concat([first, other], ignore_index=True)
if len(samples) != 3:
    raise RuntimeError('Need two distinct impressions of the first training finger.')

def read_xyt(path):
    points = []
    for line in path.read_text(encoding='ascii').splitlines():
        if not line.strip():
            continue
        values = [int(value) for value in line.split()]
        if len(values) != 4:
            raise RuntimeError(f'Unexpected MINDTCT output row: {line}')
        points.append(values)
    return points

extraction_rows = []
for index, record in enumerate(samples.to_dict('records')):
    source = DATA_ROOT / record['relative_path']
    if not source.is_file():
        raise FileNotFoundError('Runtime images are missing. Rerun notebook 02 to restore them from Drive.')
    if file_digest(source) != record['sha256']:
        raise RuntimeError(f'Source image changed since audit: {source}')
    png_path = WORK_DIR / f'sample_{index}.png'
    output_root = WORK_DIR / f'sample_{index}'
    with Image.open(source) as image:
        image.convert('L').save(png_path, dpi=(500, 500))
    command = [str(MINDTCT), str(png_path), str(output_root)]
    result = subprocess.run(command, capture_output=True, text=True, timeout=120)
    log_text = result.stdout + result.stderr
    (REPORT_DIR / f'sample_{index}_mindtct.log').write_text(log_text, encoding='utf-8')
    if result.returncode:
        raise RuntimeError(f'MINDTCT failed ({result.returncode}): {source}\n{log_text}')
    xyt_path = output_root.with_suffix('.xyt')
    if not xyt_path.is_file():
        raise RuntimeError(f'MINDTCT returned success but no template: {xyt_path}')
    points = read_xyt(xyt_path)
    extraction_rows.append({
        'sample': index, 'identity_key': record['identity_key'],
        'impression_id': record['impression_id'], 'relative_path': record['relative_path'],
        'source_sha256': record['sha256'], 'minutiae': len(points),
        'status': 'ok' if len(points) >= 10 else 'insufficient_minutiae',
        'png_path': str(png_path), 'xyt_path': str(xyt_path),
        'template_sha256': file_digest(xyt_path),
    })
    shutil.copyfile(xyt_path, REPORT_DIR / xyt_path.name)
extraction = pd.DataFrame(extraction_rows)
extraction.to_csv(REPORT_DIR / 'smoke_extraction.csv', index=False)
print(extraction[['sample', 'identity_key', 'impression_id', 'minutiae', 'status']].to_string(index=False))
if not extraction['status'].eq('ok').all():
    raise RuntimeError('Smoke sample has fewer than 10 minutiae. Logs saved; investigate before batch evaluation. Do not silently drop baseline failures.')

In [ ]:
pair_definitions = [('self', 0, 0), ('genuine', 0, 1), ('impostor', 0, 2)]
score_rows = []
for kind, probe, gallery in pair_definitions:
    command = [str(BOZORTH3), extraction.iloc[probe]['xyt_path'], extraction.iloc[gallery]['xyt_path']]
    result = subprocess.run(command, capture_output=True, text=True, timeout=120)
    (REPORT_DIR / f'{kind}_bozorth3.log').write_text(result.stdout + result.stderr, encoding='utf-8')
    if result.returncode or re.fullmatch(r'\s*\d+\s*', result.stdout) is None:
        raise RuntimeError(f'Invalid BOZORTH3 result: code={result.returncode}\n{result.stdout}\n{result.stderr}')
    score_rows.append({'pair_type': kind, 'probe_sample': probe, 'gallery_sample': gallery,
                       'score': int(result.stdout.strip())})
scores = pd.DataFrame(score_rows)
scores.to_csv(REPORT_DIR / 'smoke_scores.csv', index=False)
print(scores.to_string(index=False))
if scores.loc[scores['pair_type'].eq('self'), 'score'].iloc[0] <= 0:
    raise RuntimeError('Self-match did not score above zero. Investigate extraction/matcher compatibility.')
print('Genuine > impostor is not asserted from one pair. This is not an accuracy estimate.')

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(12, 4))
for axis, record in zip(axes, extraction.to_dict('records')):
    with Image.open(record['png_path']) as image:
        axis.imshow(image, cmap='gray', vmin=0, vmax=255)
        height = image.height
    points = read_xyt(Path(record['xyt_path']))
    axis.scatter([point[0] for point in points], [height - 1 - point[1] for point in points],
                 s=16, facecolors='none', edgecolors='red', linewidths=0.6)
    axis.set_title(f"sample {record['sample']} | {record['minutiae']} minutiae")
    axis.axis('off')
figure.tight_layout()
figure.savefig(REPORT_DIR / 'smoke_minutiae.png', dpi=150)
plt.show()
completion = {
    'run_id': RUN_ID, 'status': 'smoke_passed',
    'split_protocol_sha256': file_digest(protocol_path),
    'nbis_source_sha256': NBIS_SHA256,
    'mindtct_flags': [], 'bozorth3_flags': [],
    'image_policy': 'FVC2000 DB1 training only; L-mode PNG 500 dpi; no resizing or enhancement',
    'score_semantics': 'nonnegative similarity; higher means more similar; no threshold selected',
    'not_an_accuracy_estimate': True,
    'report_hashes': {path.name: file_digest(path) for path in REPORT_DIR.iterdir() if path.is_file()},
}
(REPORT_DIR / 'completion.json').write_text(json.dumps(completion, indent=2), encoding='utf-8')
print('NBIS SMOKE TEST PASSED. Reports saved:', REPORT_DIR)
print('Next notebook: freeze pair lists and measure original-image validation baseline.')